# Annotation script to annotate collected rollouts

In [1]:
import sys
import os
import json
from pprint import pprint
import yaml
import pathlib
import glob
import pandas as pd
import openai
from pathlib import Path

In [2]:
# Replace 'experiment_log.json' with the path to your actual JSON file
ROOT_PATH = pathlib.Path("__file__").resolve().parent.parent
EXP_FOLDER = os.path.join(ROOT_PATH, "experiments")
# Define the experiment ID
experiment_id = 'IT2_scene2_4.1_v1_pair' # Replace with your actual experiment ID
logs_id = 'IT1_scene2_4.1_v3_cp1_pair_eval' # last iteration of collected logs from which the new model learn from
OUTPUT_PATH = os.path.join(EXP_FOLDER, experiment_id)
LOGS_FOLDER = os.path.join(ROOT_PATH, "logs")
CONFIGS_FOLDER = os.path.join(ROOT_PATH, "cos_eor", "configs", "local")
ENVS_FILE_PATH = os.path.join(CONFIGS_FOLDER , "envs_demo.yaml")


# Note: put the OpenAI key here:
with open(os.path.join(CONFIGS_FOLDER, "api_key.yaml")) as kfile:
    k = yaml.safe_load(kfile)
openai.api_key = k['key'] # PUT THE API_KEY into key.txt file
if 'organization' in k:
    openai.organization = k['organization']
FINETUNE_MODEL = "ft:gpt-4.1-mini-2025-04-14:rhlab:it1-scene2-v3:D1FSrnSL:ckpt-step-55" # the base model from the last iter
SUFFIX = f"IT2_scene2_v1" # suffix to be used for the new output fine-tune model, e.g., ft_23_from_bt


TRAIN_FILE_NAME_OUTPUT = "train.jsonl"
VALID_FILE_NAME_OUTPUT = "valid.jsonl"
TEST_FILE_NAME_OUTPUT = "test.jsonl"
META_FILE_NAME = "info.txt"

# Constants
ANNOTATION = "annotation"
DIFF_CORRECT_LOC = "diff_correct_loc"
EXPERIMENT = "experiment"
FLAG = "flag"
FINETUNE_MSG = "finetune_message"
NUM_OBJECTS_DISCOVERED = "num_objects_discovered"
NUM_RECS_DISCOVERED = "num_recs_discovered"
OUTCOME = "outcome"
PROMPT = "prompt"
REWARD = "reward"
REWARD_WEIGHTS = {NUM_OBJECTS_DISCOVERED: 1, NUM_RECS_DISCOVERED: 1, DIFF_CORRECT_LOC: 10}
SCENE = "scene"
SUC = "succeeded"
SUC_STEPS = "successful_steps"

# Read the scene IDs from the envs.yaml file
with open(ENVS_FILE_PATH , 'r') as file:
    scenes = yaml.safe_load(file).split()

In [3]:
# Load the scene IDs from the envs.yaml file
def load_scenes(file_path):
    with open(file_path, 'r') as file:
        return yaml.safe_load(file)

# Function to get log file paths for a given experiment ID and scene ID
def get_experiment_log_paths(logs_id, scene_id, mode='train'):
    logs_path_pattern = f'{LOGS_FOLDER}/{logs_id}/demo/{scene_id}/data_*.json'
    all_paths = sorted(glob.glob(logs_path_pattern), reverse=False) # sorted ascending according to time
    # we need to get the train episodes for train, test episodes for evaluation
    log_paths = []
    if mode == 'all':
        return all_paths
    if mode == 'train':
        if 'ablationlarge' in logs_id:
            MOD_VAL = 35
        elif 'ablationsmall' in logs_id:
            MOD_VAL = 10
        elif 'small' in logs_id:
            MOD_VAL = 10
        elif "large" in logs_id:
            MOD_VAL = 25
        elif "pair" in logs_id:
            MOD_VAL = 25
        else:
            MOD_VAL = 10
            print ('no experiment size provided, assume small experiment')
        print ('mod val', MOD_VAL)
        for i, log in enumerate(all_paths):
            if i % MOD_VAL >= 5:
                # log 5 - 9 are for training
                log_paths.append(log)
            else:
                # log 0 - 4 are for testing
                continue
    return log_paths

def reward(result):
    reward = sum(result[k] * REWARD_WEIGHTS[k] for k in REWARD_WEIGHTS)
    return reward

def compile_steps(steps):
    # Enumerate over the steps, starting at 1, and format them into a string
    nl = "\n".join(f"step {index}: {step}" for index, step in enumerate(steps, start=1))
    nl_without_prefix = nl.replace('step 1: ', '')
    return nl_without_prefix

def prepare_example_conversation(system_msg, user_msg, assistant_message):
    messages = []
    messages.append({"role": "system", "content": system_msg,})
    messages.append({"role": "user", "content": user_msg})
    messages.append({"role": "assistant", "content": assistant_message})
    return {"messages": messages}

def annotate_record(record, logs_id, scene_id):
    result = {}
    result[SCENE] = scene_id
    result[EXPERIMENT] = logs_id
    result[NUM_OBJECTS_DISCOVERED] = len(record[OUTCOME]["objects_discovered"])
    result[NUM_RECS_DISCOVERED] = len(record[OUTCOME]["recs_discovered"])
    result[DIFF_CORRECT_LOC] = record[OUTCOME]["count_correct"]["end"] - record[OUTCOME]["count_correct"]["start"]
    # craft a response based on successful steps
    result[SUC_STEPS] = [l['step_raw'] for l in record["logs"] if l[FLAG] == SUC]
    system_msg = record["low_level"]["prompt"]["system"]
    user_msg = record["low_level"]["prompt"]["user"]
    assistant_msg = compile_steps(result[SUC_STEPS])
    result[FINETUNE_MSG] = prepare_example_conversation(system_msg=system_msg, user_msg=user_msg, assistant_message=assistant_msg)
    result[REWARD] = reward(result)
    return result

# Function to load experiment logs and add the scene name
def load_and_annotate_logs(logs_id, scenes):
    all_records = []
    all_episodes = []
    for scene_id in scenes:
        log_file_paths = get_experiment_log_paths(logs_id, scene_id, mode='train')
        for log_file_path in log_file_paths:
            with open(log_file_path, 'r') as file:
                records = json.load(file)
                # Annotate each record with the scene name
                for record in records:
                    record[ANNOTATION] = annotate_record(record, logs_id, scene_id)
                all_records.extend(records)
                all_episodes.append(records)
    return all_records, all_episodes

# Load and annotate logs
annotated_logs, annotated_episodes = load_and_annotate_logs(logs_id, scenes)


mod val 25


In [4]:
positive_records = [log for log in annotated_logs if log['annotation']['diff_correct_loc'] > 0]
pick_records = [log for log in annotated_logs if log['annotation']]

from collections import Counter
Counter([log[ANNOTATION][SCENE] for log in positive_records])

Counter({'merom_1_int': 184})

In [5]:
correct_episode_steps = []
for ie, episode in enumerate(annotated_episodes):
    flag_correct = False
    for ir, record in enumerate(episode):
        if record[ANNOTATION]['diff_correct_loc'] > 0:
            correct_episode_steps.append((ie, ir)) 

In [6]:
from pprint import pprint
print ('Number of correct step episodes', len(correct_episode_steps))
print ('Avg correct steps per episode', len(correct_episode_steps)/len(annotated_episodes))
print (correct_episode_steps)

Number of correct step episodes 184
Avg correct steps per episode 1.84
[(3, 6), (3, 15), (5, 0), (5, 11), (5, 12), (5, 14), (5, 16), (5, 18), (5, 22), (7, 13), (7, 14), (7, 21), (8, 0), (8, 2), (8, 5), (8, 8), (9, 0), (9, 2), (9, 3), (9, 9), (9, 10), (10, 9), (10, 14), (10, 17), (11, 13), (12, 5), (12, 8), (14, 4), (14, 8), (14, 10), (15, 1), (15, 6), (16, 19), (17, 12), (17, 13), (18, 6), (19, 5), (19, 6), (19, 9), (21, 4), (21, 5), (21, 10), (21, 12), (23, 8), (24, 9), (24, 11), (24, 20), (24, 22), (25, 0), (25, 4), (25, 6), (25, 10), (25, 11), (25, 13), (27, 3), (28, 0), (28, 2), (28, 5), (28, 7), (28, 12), (29, 7), (30, 7), (30, 8), (30, 9), (31, 9), (31, 12), (32, 2), (32, 6), (32, 14), (32, 19), (34, 3), (34, 4), (34, 7), (34, 11), (36, 6), (36, 7), (36, 9), (43, 11), (43, 13), (43, 15), (43, 17), (43, 25), (44, 8), (44, 10), (44, 16), (45, 0), (45, 9), (45, 12), (45, 13), (45, 17), (48, 0), (48, 3), (48, 8), (48, 9), (49, 3), (49, 4), (49, 5), (50, 2), (50, 3), (50, 5), (51, 7),

In [7]:
EPISODE_INDEX = 0
STEP_INDEX = 1
pprint (annotated_episodes[EPISODE_INDEX][STEP_INDEX][ANNOTATION][SCENE])
pprint (annotated_episodes[EPISODE_INDEX][STEP_INDEX][ANNOTATION]['successful_steps'])
pprint (annotated_episodes[EPISODE_INDEX][STEP_INDEX]['correct_objects'])

'merom_1_int'
[]
{'end': {'diaper pack 1': 'bathroom 0 top cabinet 70',
         'soap dispenser 1': 'childs room 0 bottom cabinet 2',
         'sponge 1': 'bedroom 0 bottom cabinet 12',
         'tampons 1': 'bathroom 0 top cabinet 70',
         'towel 1': 'kitchen 0 bottom cabinet 49'},
 'start': {'diaper pack 1': 'bathroom 0 top cabinet 70',
           'soap dispenser 1': 'childs room 0 bottom cabinet 2',
           'sponge 1': 'bedroom 0 bottom cabinet 12',
           'tampons 1': 'bathroom 0 top cabinet 70',
           'towel 1': 'kitchen 0 bottom cabinet 49'}}


In [8]:
finetune_records = [log for log in annotated_logs if log[ANNOTATION][DIFF_CORRECT_LOC] > 0]
print ('finetune records', len(finetune_records))
finetune_records[0][ANNOTATION][FINETUNE_MSG]

finetune records 184


{'messages': [{'role': 'system',
   'content': 'You are a one-handed household robot.'},
  {'role': 'user',
   'content': 'There are objects misplaced on wrong receptacles and potentially in the wrong room. \nYou are holding dish drainer 1.\nIn bedroom 0, no receptacles found yet. \nIn bathroom 0, no receptacles found yet. \nIn childs room 0, no receptacles found yet. \nIn living room 0, found receptacles: living room 0 carpet 62, living room 0 coffee table 27, living room 0 floor lamp 31, living room 0 floor lamp 33, living room 0 mirror 68, living room 0 sofa chair 26, living room 0 towel rack 66, living room 0 bottom cabinet 30, living room 0 carpet 28. \nIn dining room 0, found receptacles: dining room 0 chair 22, dining room 0 table 20, dining room 0 chair 21, dining room 0 chair 24, dining room 0 plant 29, dining room 0 chair 23. \nIn kitchen 0, found receptacles: kitchen 0 bottom cabinet 50, kitchen 0 top cabinet 55, kitchen 0 bottom cabinet 49, kitchen 0 sink 53, kitchen 0 top 

## remove duplicates
(keep the first of groupby same 'object_moved', 'rec_before', 'rec_after', 'correct_before')

In [9]:
df_all = pd.DataFrame(annotated_logs)
df_all['object_moved'] = df_all.apply(lambda x: list(x['outcome']['objects_moved'].keys())[0] if len(x['outcome']['objects_moved'].keys()) > 0 else None, axis=1)
df_all['rec_before'] = df_all.apply(lambda x: list(x['outcome']['objects_moved'].values())[0][0] if len(x['outcome']['objects_moved'].keys()) > 0 else None, axis=1)
df_all['rec_after'] = df_all.apply(lambda x: list(x['outcome']['objects_moved'].values())[0][1] if len(x['outcome']['objects_moved'].keys()) > 0 else None, axis=1)
df_all['correct_before'] = df_all.apply(lambda x: str(set(x['correct_objects']['start'])), axis=1)
df_all_remove_duplicate = df_all.groupby(['object_moved', 'rec_before', 'rec_after', 'correct_before']).first().reset_index()


In [10]:
# let's see if we could de-duplicate before merging
df_finetune = pd.DataFrame(finetune_records)
df_finetune['object_moved'] = df_finetune.apply(lambda x: list(x['outcome']['objects_moved'].keys())[0], axis=1)
df_finetune['rec_before'] = df_finetune.apply(lambda x: list(x['outcome']['objects_moved'].values())[0][0], axis=1)
df_finetune['rec_after'] = df_finetune.apply(lambda x: list(x['outcome']['objects_moved'].values())[0][1], axis=1)
df_finetune['correct_before'] = df_finetune.apply(lambda x: str(set(x['correct_objects']['start'])), axis=1)
df_remove_duplicate = df_finetune.groupby(['object_moved', 'rec_before', 'rec_after', 'correct_before']).first().reset_index()
finetune_msg_simulation = df_remove_duplicate.apply(lambda x: x[ANNOTATION][FINETUNE_MSG], axis=1).to_list()
print ('all finetune records before duplicate removal', len(df_finetune))
print ('finetune records after duplicate removal', len(finetune_msg_simulation))
print ('num correct objects', len(df_finetune.groupby(['object_moved']).first()))

all finetune records before duplicate removal 184
finetune records after duplicate removal 169
num correct objects 40


In [11]:
raise Exception('move forward to write messages to jsonl')

Exception: move forward to write messages to jsonl

In [12]:
finetune_records = finetune_msg_simulation # [log for log in annotated_logs if log[ANNOTATION][DIFF_CORRECT_LOC] > 0]
NUM_VALID = 0
NUM_TRAIN = len(finetune_records) - NUM_VALID
print ('all positive records', len(finetune_records))
training_data = finetune_records[:NUM_TRAIN]# [log[ANNOTATION][FINETUNE_MSG] for log in finetune_records[:400]]
validation_data = finetune_records[NUM_TRAIN:] # [log[ANNOTATION][FINETUNE_MSG] for log in finetune_records[400:]]

all positive records 169


In [13]:
def write_jsonl(data_list: list, filename: str) -> None:
    with open(filename, "w") as out:
        for ddict in data_list:
            jout = json.dumps(ddict) + "\n"
            out.write(jout)
            
def write_metadata_file(filepath):
    with open(filepath, 'w') as out:
        flag = ""
        flag += f"Number of training samples: {len(training_data)}\n"
        flag += f"Number of validation samples: {len(validation_data)}\n"
        flag += f"Source folder: {logs_id}"
        out.write(flag)
        print (flag)

In [14]:
training_file_name = os.path.join(OUTPUT_PATH, TRAIN_FILE_NAME_OUTPUT)
# Create the parent directory if it doesn't exist
Path(training_file_name).parent.mkdir(parents=True, exist_ok=True)
write_jsonl(training_data, training_file_name)

if NUM_VALID > 0:
    validation_file_name = os.path.join(OUTPUT_PATH, VALID_FILE_NAME_OUTPUT)
    write_jsonl(validation_data, validation_file_name)

write_metadata_file(os.path.join(OUTPUT_PATH, META_FILE_NAME))


Number of training samples: 169
Number of validation samples: 0
Source folder: IT1_scene2_4.1_v3_cp1_pair_eval


In [15]:
raise Exception('move forward to create finetune jobs')

Exception: move forward to create finetune jobs

In [16]:
def create_finetune_upload_response():
    training_response = openai.File.create(file=open(training_file_name, "rb"), purpose="fine-tune")
    training_file_id = training_response["id"]
    print("Training file ID:", training_file_id)
    if NUM_VALID > 0:
        validation_response = openai.File.create(file=open(validation_file_name, "rb"), purpose="fine-tune")
        validation_file_id = validation_response["id"]
        print("Validation file ID:", validation_file_id)
        return {"training_response": training_response, "validation_response": validation_response}
    else:
        return {"training_response": training_response}

def create_finetune_response_and_log(training_file_id, validation_file_id=None):
    if validation_file_id is not None:
        response = openai.FineTuningJob.create( training_file=training_file_id, validation_file=validation_file_id, model=FINETUNE_MODEL, suffix=SUFFIX, \
            hyperparameters={"n_epochs":2, "batch_size":2, "learning_rate_multiplier":0.9})
    else:
        response = openai.FineTuningJob.create( training_file=training_file_id, model=FINETUNE_MODEL, suffix=SUFFIX, \
            hyperparameters={"n_epochs":2, "batch_size":2, "learning_rate_multiplier":0.9})
    job_id = response["id"]
    with open(os.path.join(OUTPUT_PATH, "job_info.txt"), 'w') as out:
        flag = ""
        flag += f"training file id: {training_file_id}\n"
        flag += f"validation file id: {validation_file_id}\n"
        flag += f"finetune job id: {job_id}\n"
        flag += f"finetune model: {FINETUNE_MODEL}\n"
        flag += f"finetune suffix: {SUFFIX}"
        out.write(flag)
        print("Status:", response["status"])
        print("Job ID:", response["id"])
    return response

In [17]:
upload_result = create_finetune_upload_response()
training_response = upload_result['training_response']
validation_file_id = None
if NUM_VALID > 0:
    validation_response = upload_result['validation_response']
    validation_file_id = validation_response['id']
training_file_id = training_response['id']

finetune_response = create_finetune_response_and_log(training_file_id, validation_file_id)
job_id = finetune_response['id']

Training file ID: file-YERi4nTBbXs8vdnjhzAzMo
Status: validating_files
Job ID: ftjob-otsXGIVZc8TlQ3fyAAsoveih


In [18]:
response = openai.FineTuningJob.retrieve(job_id)

print("Job ID:", response["id"])
print("Status:", response["status"])
print("Trained Tokens:", response["trained_tokens"])

Job ID: ftjob-GgWPKtZ3UCpvShmUqJPSOqvV
Status: validating_files
Trained Tokens: None
